In [1]:
# Day 4 — groupBy, agg, Aggregate Functions

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, count, max, min,
    round, countDistinct, collect_list, collect_set
)

spark = SparkSession.builder.appName("Day4").getOrCreate()

data = [
    ("Ravi",   "Engineering", "Pune",      55000, "M", 28),
    ("Priya",  "HR",          "Mumbai",    42000, "F", 32),
    ("Arjun",  "Engineering", "Delhi",     72000, "M", 26),
    ("Sneha",  "Finance",     "Pune",      61000, "F", 30),
    ("Rohit",  "Engineering", "Mumbai",    80000, "M", 35),
    ("Meera",  "HR",          "Bangalore", 39000, "F", 27),
    ("Karan",  "Finance",     "Delhi",     55000, "M", 29),
    ("Divya",  "Engineering", "Pune",      91000, "F", 33),
    ("Nitin",  "HR",          "Mumbai",    44000, "M", 31),
    ("Anjali", "Finance",     "Bangalore", 67000, "F", 28),
    ("Rahul",  "Engineering", "Delhi",     68000, "M", 30),
    ("Pooja",  "HR",          "Pune",      41000, "F", 26),
]

cols = ["name", "dept", "city", "salary", "gender", "age"]
df = spark.createDataFrame(data, cols)
df.show()

+------+-----------+---------+------+------+---+
|  name|       dept|     city|salary|gender|age|
+------+-----------+---------+------+------+---+
|  Ravi|Engineering|     Pune| 55000|     M| 28|
| Priya|         HR|   Mumbai| 42000|     F| 32|
| Arjun|Engineering|    Delhi| 72000|     M| 26|
| Sneha|    Finance|     Pune| 61000|     F| 30|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|
| Meera|         HR|Bangalore| 39000|     F| 27|
| Karan|    Finance|    Delhi| 55000|     M| 29|
| Divya|Engineering|     Pune| 91000|     F| 33|
| Nitin|         HR|   Mumbai| 44000|     M| 31|
|Anjali|    Finance|Bangalore| 67000|     F| 28|
| Rahul|Engineering|    Delhi| 68000|     M| 30|
| Pooja|         HR|     Pune| 41000|     F| 26|
+------+-----------+---------+------+------+---+



In [3]:
df


DataFrame[name: string, dept: string, city: string, salary: bigint, gender: string, age: bigint]

# TASK 1 — Basic groupBy + single aggregation
groupBy() is a func groups rows that have the same value in one or more columns.
After grouping, you usually perform an aggregation such as:

Count
Sum
Average
Maximum
Minimum

Count employees per department
df.groupBy("dept").count().show()


In [4]:
df.groupBy("dept").count().show()

+-----------+-----+
|       dept|count|
+-----------+-----+
|Engineering|    5|
|         HR|    4|
|    Finance|    3|
+-----------+-----+



In [5]:
df.show(5)

+-----+-----------+------+------+------+---+
| name|       dept|  city|salary|gender|age|
+-----+-----------+------+------+------+---+
| Ravi|Engineering|  Pune| 55000|     M| 28|
|Priya|         HR|Mumbai| 42000|     F| 32|
|Arjun|Engineering| Delhi| 72000|     M| 26|
|Sneha|    Finance|  Pune| 61000|     F| 30|
|Rohit|Engineering|Mumbai| 80000|     M| 35|
+-----+-----------+------+------+------+---+
only showing top 5 rows



In [6]:
df.groupBy("dept").sum("salary").show()

+-----------+-----------+
|       dept|sum(salary)|
+-----------+-----------+
|Engineering|     366000|
|         HR|     166000|
|    Finance|     183000|
+-----------+-----------+



In [7]:
df.groupBy("dept").avg("salary").show()

+-----------+-----------+
|       dept|avg(salary)|
+-----------+-----------+
|Engineering|    73200.0|
|         HR|    41500.0|
|    Finance|    61000.0|
+-----------+-----------+



In [8]:
# 1d. Max and Min salary per city
df.groupBy("city").max("salary").show()
df.groupBy("city").min("salary").show()

+---------+-----------+
|     city|max(salary)|
+---------+-----------+
|     Pune|      91000|
|   Mumbai|      80000|
|    Delhi|      72000|
|Bangalore|      67000|
+---------+-----------+

+---------+-----------+
|     city|min(salary)|
+---------+-----------+
|     Pune|      41000|
|   Mumbai|      42000|
|    Delhi|      55000|
|Bangalore|      39000|
+---------+-----------+



TASK 2 — agg() for multiple aggregations at once
# 2a. Multiple aggregations in one shot — this is the real-world way
df.groupBy("dept").agg(
    count("name").alias("employee_count"),
    round(avg("salary"), 2).alias("avg_salary"),
    sum("salary").alias("total_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
).show()

# 2b. Group by multiple columns
df.groupBy("dept", "city").agg(
    count("name").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary")
).orderBy("dept", "city").show()

# 2c. Group by gender
df.groupBy("gender").agg(
    count("*").alias("total"),
    round(avg("salary"), 2).alias("avg_salary"),
    max("age").alias("max_age")
).show()

In [9]:
df.groupBy("dept").agg(
    count("name").alias("employee_count"),
    round(avg("salary"), 2).alias("avg_salary"),
    sum("salary").alias("total_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
).show()

+-----------+--------------+----------+------------+----------+----------+
|       dept|employee_count|avg_salary|total_salary|max_salary|min_salary|
+-----------+--------------+----------+------------+----------+----------+
|Engineering|             5|   73200.0|      366000|     91000|     55000|
|         HR|             4|   41500.0|      166000|     44000|     39000|
|    Finance|             3|   61000.0|      183000|     67000|     55000|
+-----------+--------------+----------+------------+----------+----------+



In [10]:
df.groupBy("dept", "city").agg(
    count("name").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary")
).orderBy("dept", "city").show()

+-----------+---------+---------+----------+
|       dept|     city|headcount|avg_salary|
+-----------+---------+---------+----------+
|Engineering|    Delhi|        2|   70000.0|
|Engineering|   Mumbai|        1|   80000.0|
|Engineering|     Pune|        2|   73000.0|
|    Finance|Bangalore|        1|   67000.0|
|    Finance|    Delhi|        1|   55000.0|
|    Finance|     Pune|        1|   61000.0|
|         HR|Bangalore|        1|   39000.0|
|         HR|   Mumbai|        2|   43000.0|
|         HR|     Pune|        1|   41000.0|
+-----------+---------+---------+----------+



In [11]:
df.groupBy("dept").count.show()

AttributeError: 'function' object has no attribute 'show'

In [12]:
df.groupBy("gender").agg(
    count("*").alias("total"),
    round(avg("salary"), 0).alias("avg_salary"),
    max("age").alias("max_age")
).show()

+------+-----+----------+-------+
|gender|total|avg_salary|max_age|
+------+-----+----------+-------+
|     M|    6|   62333.0|     35|
|     F|    6|   56833.0|     33|
+------+-----+----------+-------+



✅ TASK 3 — countDistinct + collect_list + collect_set

# 3a. Count distinct cities per department
df.groupBy("dept").agg(
    countDistinct("city").alias("unique_cities")
).show()

# 3b. collect_list — list all employee names per dept (with duplicates)
df.groupBy("dept").agg(
    collect_list("name").alias("all_employees")
).show(truncate=False)

# 3c. collect_set — unique cities per dept (no duplicates)
df.groupBy("dept").agg(
    collect_set("city").alias("office_locations")
).show(truncate=False)

In [13]:
df.groupBy("dept").agg(
    countDistinct("city").alias("unique_cities")
).show()

+-----------+-------------+
|       dept|unique_cities|
+-----------+-------------+
|Engineering|            3|
|         HR|            3|
|    Finance|            3|
+-----------+-------------+



In [14]:
df.show()

+------+-----------+---------+------+------+---+
|  name|       dept|     city|salary|gender|age|
+------+-----------+---------+------+------+---+
|  Ravi|Engineering|     Pune| 55000|     M| 28|
| Priya|         HR|   Mumbai| 42000|     F| 32|
| Arjun|Engineering|    Delhi| 72000|     M| 26|
| Sneha|    Finance|     Pune| 61000|     F| 30|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|
| Meera|         HR|Bangalore| 39000|     F| 27|
| Karan|    Finance|    Delhi| 55000|     M| 29|
| Divya|Engineering|     Pune| 91000|     F| 33|
| Nitin|         HR|   Mumbai| 44000|     M| 31|
|Anjali|    Finance|Bangalore| 67000|     F| 28|
| Rahul|Engineering|    Delhi| 68000|     M| 30|
| Pooja|         HR|     Pune| 41000|     F| 26|
+------+-----------+---------+------+------+---+



In [15]:
df.select("city").distinct().show()

+---------+
|     city|
+---------+
|     Pune|
|   Mumbai|
|    Delhi|
|Bangalore|
+---------+



In [16]:
df.select("city").distinct().orderBy("city") .show()

+---------+
|     city|
+---------+
|Bangalore|
|    Delhi|
|   Mumbai|
|     Pune|
+---------+



In [17]:
df.select("city").distinct().count()

4

In [18]:
# Count employees in each city
df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|     Pune|    4|
|   Mumbai|    3|
|    Delhi|    3|
|Bangalore|    2|
+---------+-----+



In [19]:
# 3b. collect_list — list all employee names per dept (with duplicates)
df.groupBy("dept").agg(
    collect_list("name").alias("all_employees")
).show(truncate=False)

+-----------+----------------------------------+
|dept       |all_employees                     |
+-----------+----------------------------------+
|Engineering|[Ravi, Arjun, Rohit, Divya, Rahul]|
|HR         |[Priya, Meera, Nitin, Pooja]      |
|Finance    |[Sneha, Karan, Anjali]            |
+-----------+----------------------------------+



In [20]:
# 3c. collect_set — unique cities per dept (no duplicates)
df.groupBy("dept").agg(
    collect_set("city").alias("office_locations")
).show(truncate=False)

+-----------+-------------------------+
|dept       |office_locations         |
+-----------+-------------------------+
|Engineering|[Pune, Delhi, Mumbai]    |
|HR         |[Bangalore, Pune, Mumbai]|
|Finance    |[Bangalore, Pune, Delhi] |
+-----------+-------------------------+



TASK 4 — Filter AFTER groupBy using having style
# In PySpark, HAVING = filter() after agg()

# 4a. Show only departments with avg salary > 55000
df.groupBy("dept").agg(
    round(avg("salary"), 2).alias("avg_salary")
).filter(col("avg_salary") > 55000).show()

# 4b. Show cities with more than 2 employees
df.groupBy("city").agg(
    count("name").alias("emp_count")
).filter(col("emp_count") > 2).show()

# 4c. Departments where total salary bill > 150000
df.groupBy("dept").agg(
    sum("salary").alias("total_salary_bill")
).filter(col("total_salary_bill") > 150000) \
 .orderBy(col("total_salary_bill").desc()) \
 .show()

In [21]:
# 4a. Show only departments with avg salary > 55000
df.groupBy("dept").agg( avg("salary")

SyntaxError: incomplete input (3728976044.py, line 2)

In [22]:
spark.stop()

In [23]:
spark

AttributeError: 'NoneType' object has no attribute 'sc'

In [24]:
spark.range(5).show()

AttributeError: 'NoneType' object has no attribute 'sc'

In [25]:
df


DataFrame[name: string, dept: string, city: string, salary: bigint, gender: string, age: bigint]

In [28]:
df.show()

+------+-----------+---------+------+------+---+
|  name|       dept|     city|salary|gender|age|
+------+-----------+---------+------+------+---+
|  Ravi|Engineering|     Pune| 55000|     M| 28|
| Priya|         HR|   Mumbai| 42000|     F| 32|
| Arjun|Engineering|    Delhi| 72000|     M| 26|
| Sneha|    Finance|     Pune| 61000|     F| 30|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|
| Meera|         HR|Bangalore| 39000|     F| 27|
| Karan|    Finance|    Delhi| 55000|     M| 29|
| Divya|Engineering|     Pune| 91000|     F| 33|
| Nitin|         HR|   Mumbai| 44000|     M| 31|
|Anjali|    Finance|Bangalore| 67000|     F| 28|
| Rahul|Engineering|    Delhi| 68000|     M| 30|
| Pooja|         HR|     Pune| 41000|     F| 26|
+------+-----------+---------+------+------+---+



TASK 3 — countDistinct + collect_list + collect_set
# 3a. Count distinct cities per department
df.groupBy("dept").agg(
    countDistinct("city").alias("unique_cities")
).show()

# 3b. collect_list — list all employee names per dept (with duplicates)
df.groupBy("dept").agg(
    collect_list("name").alias("all_employees")
).show(truncate=False)

# 3c. collect_set — unique cities per dept (no duplicates)
df.groupBy("dept").agg(
    collect_set("city").alias("office_locations")
).show(truncate=False)

In [29]:
df.groupBy("dept").agg(
    countDistinct("city").alias("unique_cities")
).show()

+-----------+-------------+
|       dept|unique_cities|
+-----------+-------------+
|Engineering|            3|
|         HR|            3|
|    Finance|            3|
+-----------+-------------+



In [30]:
df.groupBy("dept").agg(
    collect_list("name").alias("all_employees")
).show(truncate=False)

+-----------+----------------------------------+
|dept       |all_employees                     |
+-----------+----------------------------------+
|Engineering|[Ravi, Arjun, Rohit, Divya, Rahul]|
|HR         |[Priya, Meera, Nitin, Pooja]      |
|Finance    |[Sneha, Karan, Anjali]            |
+-----------+----------------------------------+



In [31]:
df.groupBy("dept").agg(
    collect_set("city").alias("office_locations")
).show(truncate=False)

+-----------+-------------------------+
|dept       |office_locations         |
+-----------+-------------------------+
|Engineering|[Pune, Delhi, Mumbai]    |
|HR         |[Bangalore, Pune, Mumbai]|
|Finance    |[Bangalore, Pune, Delhi] |
+-----------+-------------------------+



TASK 4 — Filter AFTER groupBy using having style
# In PySpark, HAVING = filter() after agg()

# 4a. Show only departments with avg salary > 55000
df.groupBy("dept").agg(
    round(avg("salary"), 2).alias("avg_salary")
).filter(col("avg_salary") > 55000).show()

# 4b. Show cities with more than 2 employees
df.groupBy("city").agg(
    count("name").alias("emp_count")
).filter(col("emp_count") > 2).show()

# 4c. Departments where total salary bill > 150000
df.groupBy("dept").agg(
    sum("salary").alias("total_salary_bill")
).filter(col("total_salary_bill") > 150000) \
 .orderBy(col("total_salary_bill").desc()) \
 .show()

In [36]:
# Show only departments with avg salary > 55000
df.groupBy("dept").agg(
    round(avg("salary"),2).alias("Avg_salary")
).filter(col("avg_salary") >55000).show()

+-----------+----------+
|       dept|Avg_salary|
+-----------+----------+
|Engineering|   73200.0|
|    Finance|   61000.0|
+-----------+----------+



In [37]:
df.groupBy("city").agg(
    count("name").alias("emp_count")
).filter(col("emp_count") > 2).show()

+------+---------+
|  city|emp_count|
+------+---------+
|  Pune|        4|
|Mumbai|        3|
| Delhi|        3|
+------+---------+



In [38]:
df.groupBy("dept").agg(
    sum("salary").alias("total_salary_bill")
).filter(col("total_salary_bill") > 150000) \
 .orderBy(col("total_salary_bill").desc()) \
 .show()

+-----------+-----------------+
|       dept|total_salary_bill|
+-----------+-----------------+
|Engineering|           366000|
|    Finance|           183000|
|         HR|           166000|
+-----------+-----------------+



✅ TASK 5 — Real World Aggregation Scenario
# Business Question:
# "Give me a department summary showing headcount,
#  avg salary, total payroll, highest paid employee city,
#  and all cities where we have employees"

dept_summary = df.groupBy("dept").agg(
    count("name").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
    sum("salary").alias("total_payroll"),
    countDistinct("city").alias("num_cities"),
    collect_set("city").alias("cities_present")
).orderBy(col("total_payroll").desc())

dept_summary.show(truncate=False)

In [39]:
df.show()

+------+-----------+---------+------+------+---+
|  name|       dept|     city|salary|gender|age|
+------+-----------+---------+------+------+---+
|  Ravi|Engineering|     Pune| 55000|     M| 28|
| Priya|         HR|   Mumbai| 42000|     F| 32|
| Arjun|Engineering|    Delhi| 72000|     M| 26|
| Sneha|    Finance|     Pune| 61000|     F| 30|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|
| Meera|         HR|Bangalore| 39000|     F| 27|
| Karan|    Finance|    Delhi| 55000|     M| 29|
| Divya|Engineering|     Pune| 91000|     F| 33|
| Nitin|         HR|   Mumbai| 44000|     M| 31|
|Anjali|    Finance|Bangalore| 67000|     F| 28|
| Rahul|Engineering|    Delhi| 68000|     M| 30|
| Pooja|         HR|     Pune| 41000|     F| 26|
+------+-----------+---------+------+------+---+



In [42]:
df_summary = df.groupBy("dept").agg(
    count("name").alias("headcount"),
    round(avg("salary"),2).alias("avg_salary"),
          sum("salary").alias("total_payroll"),
        countDistinct("city").alias("num_cities"),
    collect_set("city").alias("cities_present")
).orderBy(col("total_payroll").desc())

df_summary.show(truncate=False)

+-----------+---------+----------+-------------+----------+-------------------------+
|dept       |headcount|avg_salary|total_payroll|num_cities|cities_present           |
+-----------+---------+----------+-------------+----------+-------------------------+
|Engineering|5        |73200.0   |366000       |3         |[Pune, Delhi, Mumbai]    |
|Finance    |3        |61000.0   |183000       |3         |[Bangalore, Pune, Delhi] |
|HR         |4        |41500.0   |166000       |3         |[Bangalore, Pune, Mumbai]|
+-----------+---------+----------+-------------+----------+-------------------------+



Day 4 exe.
1. Find total salary and headcount per city

2. Find average age per department — round to 1 decimal

3. Find departments where employee count >= 3

4. Find the city with highest average salary

5. Group by dept + gender, show count and avg salary

6. Show departments where max salary > 80000

7. Business case: HR wants a report of each city showing:
   - number of employees
   - total salary cost
   - departments present (use collect_set)
   - average age

In [43]:
df.groupBy("city").agg(
    sum("salary").alias ("total_salary"),
    count("name").alias("head_count")
).show()

+---------+------------+----------+
|     city|total_salary|head_count|
+---------+------------+----------+
|     Pune|      248000|         4|
|   Mumbai|      166000|         3|
|    Delhi|      195000|         3|
|Bangalore|      106000|         2|
+---------+------------+----------+



In [44]:
df.groupBy("dept").agg(
    round(avg("age"),1).alias ("avg_age")
).show()
    

+-----------+-------+
|       dept|avg_age|
+-----------+-------+
|Engineering|   30.4|
|         HR|   29.0|
|    Finance|   29.0|
+-----------+-------+



In [47]:
df.groupBy("dept").agg(
    sum("name").alias("headcount")).filter(col("headcount")>=3).show()

+----+---------+
|dept|headcount|
+----+---------+
+----+---------+



In [49]:
#4. Find the city with highest average salary
df.groupby("city").agg(
    round(avg("salary"),0).alias("avg_salary")
).orderBy(col("avg_salary").desc()).show(1)

+-----+----------+
| city|avg_salary|
+-----+----------+
|Delhi|   65000.0|
+-----+----------+
only showing top 1 row



In [50]:
#5. Group by dept + gender, show count and avg salary
df.groupBy("dept", "gender").agg(
    count("name").alias("count"),
    round(avg("salary"), 0).alias("avg_salary")
).orderBy("dept","gender").show()

+-----------+------+-----+----------+
|       dept|gender|count|avg_salary|
+-----------+------+-----+----------+
|Engineering|     F|    1|   91000.0|
|Engineering|     M|    4|   68750.0|
|    Finance|     F|    2|   64000.0|
|    Finance|     M|    1|   55000.0|
|         HR|     F|    3|   40667.0|
|         HR|     M|    1|   44000.0|
+-----------+------+-----+----------+



In [51]:
#6. Show departments where max salary > 80000
df.groupBy("dept").agg(
    max("salary").alias("maxSalary")
).filter (col("maxSalary") > 80000).show()

+-----------+---------+
|       dept|maxSalary|
+-----------+---------+
|Engineering|    91000|
+-----------+---------+



7. Business case: HR wants a report of each city showing:
   - number of employees
   - total salary cost
   - departments present (use collect_set)
   - average age

In [53]:
df_summary = df.groupBy("city").agg(
    count("name").alias("headCount"),
    sum("salary").alias("total_salary"),
    collect_set("dept"),
    round(avg("age"),0).alias("avgAge")).show()

+---------+---------+------------+--------------------+------+
|     city|headCount|total_salary|   collect_set(dept)|avgAge|
+---------+---------+------------+--------------------+------+
|     Pune|        4|      248000|[Finance, HR, Eng...|  29.0|
|   Mumbai|        3|      166000|   [HR, Engineering]|  33.0|
|    Delhi|        3|      195000|[Finance, Enginee...|  28.0|
|Bangalore|        2|      106000|       [Finance, HR]|  28.0|
+---------+---------+------------+--------------------+------+



In [54]:
df.groupBy("city").agg(
    count("name").alias("num_employees"),
    sum("salary").alias("total_salary_cost"),
    collect_set("dept").alias("departments"),
    round(avg("age"), 1).alias("avg_age")
).orderBy(col("num_employees").desc()).show(truncate=False)

+---------+-------------+-----------------+--------------------------+-------+
|city     |num_employees|total_salary_cost|departments               |avg_age|
+---------+-------------+-----------------+--------------------------+-------+
|Pune     |4            |248000           |[Finance, HR, Engineering]|29.3   |
|Mumbai   |3            |166000           |[HR, Engineering]         |32.7   |
|Delhi    |3            |195000           |[Finance, Engineering]    |28.3   |
|Bangalore|2            |106000           |[Finance, HR]             |27.5   |
+---------+-------------+-----------------+--------------------------+-------+

